In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.features import add_spectral_features
from src.health import prepare
from src.labels import run_table, find_onset, add_labels, RUNS
from src.models_anomaly import Mahalanobis, PCAMonitor, IsoForest
from src.evaluate import evaluate_run

df = pd.read_parquet("../data/processed_features.parquet")
spec = np.load("../data/env_spectra.npy")
df = add_spectral_features(df, spec)

FEATURES = (["rms", "peak", "kurt", "skew", "crest", "shape", "impulse",
             "spec_entropy", "spec_centroid"]
            + [f"band{i}_frac" for i in range(7)])          # band7 dropped: fractions sum to 1

z_runs = prepare(df, FEATURES)                               # per-bearing z-scores, causal smoothing, burn-in removed
tables = {f"{s}_{b}": run_table(df, s, b) for s, b in RUNS}

labels = {}
for variant, thr in (("tight", 1.25), ("loose", 1.10)):
    for name, u in tables.items():
        labels[(variant, name)] = add_labels(u, find_onset(u, thr, 20)).set_index("idx")["anomaly"]

make_models = lambda: [Mahalanobis(), PCAMonitor(stat="t2"), PCAMonitor(stat="spe"), IsoForest()]

rows, scores = [], {}
for name, Z in z_runs.items():
    for m in make_models():
        res, s, thr = evaluate_run(Z, FEATURES, m, labels[("tight", name)], labels[("loose", name)],
                                   tables[name].set_index("idx")["rul_hours"])
        rows.append({"run": name, **res})
        scores[(name, m.name)] = (s, thr)
results = pd.DataFrame(rows).round(3)

for col in ["auc_tight", "auc_loose", "lead_hours", "pre_onset_alarm_rate"]:
    print(col); display(results.pivot(index="model", columns="run", values=col))

Anomaly score over time with the threshold and onsets marked.

In [ ]:
names = ["mahalanobis", "pca_t2", "pca_spe", "isolation_forest"]
fig, axes = plt.subplots(len(z_runs), 4, figsize=(18, 10))
for r, (name, Z) in enumerate(z_runs.items()):
    for c, mname in enumerate(names):
        s, thr = scores[(name, mname)]
        ax = axes[r, c]
        ax.semilogy(Z["life"], s, lw=0.6)
        ax.axhline(thr, color="r", ls="--", lw=0.8)
        ax.axvspan(0.15, 0.32, color="g", alpha=0.12)         # training window
        ax.axvspan(0.32, 0.40, color="orange", alpha=0.12)    # calibration window
        for tag, ls in (("tight", "-"), ("loose", ":")):
            y = labels[(tag, name)].reindex(Z.index).to_numpy()
            if y.sum():
                ax.axvline(Z["life"][y == 1].iloc[0], color="k", ls=ls, lw=0.8)
        ax.set_title(f"{name} | {mname}", fontsize=9)
plt.tight_layout(); plt.savefig("../results/anomaly_scores.png", dpi=150)

The control-bearing figure

In [ ]:
from src.models_anomaly import Mahalanobis
from src.evaluate import TRAIN, CAL

CONTROL = [("set1", "b1"), ("set1", "b2"), ("set3", "b1"), ("set3", "b2")]   # never failed
all_z = {**z_runs, **prepare(df, FEATURES, runs_list=CONTROL)}

def maha(Z):
    life = Z["life"].to_numpy(); X = Z[FEATURES].to_numpy()
    m = Mahalanobis().fit(X[(life >= TRAIN[0]) & (life < TRAIN[1])])
    s = m.score(X)
    thr = np.quantile(s[(life >= CAL[0]) & (life < CAL[1])], 0.995)
    return life, s / thr                 # 1.0 = alarm level, so runs are comparable

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, prefix in zip(axes, ["set1", "set3"]):
    for name, Z in all_z.items():
        if name.startswith(prefix):
            life, r = maha(Z)
            ax.semilogy(life, r, lw=0.7, label=name)
    ax.axhline(1, color="k", ls="--"); ax.axvline(0.41, color="r", ls=":", lw=0.8)
    ax.set_title(prefix); ax.legend()
plt.tight_layout(); plt.savefig("../results/control_bearings.png", dpi=150)

Set 1 B3 shift table

In [ ]:
Z = z_runs["set1_b3"]; life = Z["life"]
w = lambda lo, hi: Z.loc[(life >= lo) & (life < hi), FEATURES].median()
shift = pd.DataFrame({"0.32-0.40": w(.32, .40) - w(.15, .32),
                      "0.42-0.60": w(.42, .60) - w(.15, .32)})
shift.reindex(shift["0.42-0.60"].abs().sort_values(ascending=False).index).round(2)

Set 1 B4 shift table

In [ ]:
Z = z_runs["set1_b4"]; life = Z["life"]
w = lambda lo, hi: Z.loc[(life >= lo) & (life < hi), FEATURES].median()
shift = pd.DataFrame({"0.32-0.40": w(.32, .40) - w(.15, .32),
                      "0.42-0.60": w(.42, .60) - w(.15, .32)})
shift.reindex(shift["0.42-0.60"].abs().sort_values(ascending=False).index).round(2)

In [ ]:
results.pivot(index="model", columns="run", values="alarm_life")

Next experiment: alarms on never-failed bearings

In [ ]:
from src.evaluate import TRAIN, CAL, confirmed_alarm

CONTROL = [("set1", "b1"), ("set1", "b2"), ("set2", "b2"), ("set2", "b3"),
           ("set2", "b4"), ("set3", "b1"), ("set3", "b2")]
all_z = {**z_runs, **prepare(df, FEATURES, runs_list=CONTROL)}

def alarm_only(Z, model, q=0.995, n_consec=10):
    life = Z["life"].to_numpy(); X = Z[FEATURES].to_numpy()
    tr = (life >= TRAIN[0]) & (life < TRAIN[1])
    ca = (life >= CAL[0]) & (life < CAL[1])
    te = life >= CAL[1]
    model.fit(X[tr]); s = model.score(X)
    k = confirmed_alarm(s[te], np.quantile(s[ca], q), n_consec)
    return np.nan if k is None else round(float(life[np.flatnonzero(te)[k]]), 3)

res = pd.DataFrame({m().name: {name: alarm_only(Z, m()) for name, Z in all_z.items()}
                    for m in (Mahalanobis, lambda: PCAMonitor(stat="t2"),
                              lambda: PCAMonitor(stat="spe"), IsoForest)})
res["failed"] = [n in z_runs for n in res.index]
res.sort_values(["failed", "mahalanobis"], ascending=False)

In [ ]:
from src.evaluate import alarm_blocked

MODELS = {"mahalanobis": Mahalanobis,
          "pca_t2": lambda: PCAMonitor(stat="t2"),
          "pca_spe": lambda: PCAMonitor(stat="spe"),
          "isolation_forest": IsoForest}

def run_all(base):
    rows = []
    for name, Z in all_z.items():
        for mname, mk in MODELS.items():
            res, _, _ = alarm_blocked(Z, FEATURES, mk, base=base)
            rows.append({"run": name, "model": mname, **res})
    return pd.DataFrame(rows)

def margins(alarm):
    rows = {}
    for f in z_runs:
        ctrl = [n for n in alarm.index if n.startswith(f.split("_")[0]) and n not in z_runs]
        rows[f] = alarm.loc[ctrl].min() - alarm.loc[f]
    return pd.DataFrame(rows).T.round(3)

for base in [(0.15, 0.40), (0.15, 0.32)]:
    r = run_all(base)
    print("base window:", base)
    display(margins(r.pivot(index="run", columns="model", values="alarm_life")))
    display(r.pivot(index="run", columns="model", values="start_above").round(2))

Remove the rig-level drift

In [ ]:
CONTROL = [("set1", "b1"), ("set1", "b2"), ("set2", "b2"), ("set2", "b3"), ("set2", "b4"),
           ("set3", "b1"), ("set3", "b2"), ("set3", "b4")]
all_z = {**z_runs, **prepare(df, FEATURES, runs_list=CONTROL)}


def common_mode_removed(zdict, features):
    out = {}
    for s in ("set1", "set2", "set3"):
        names = [n for n in zdict if n.startswith(s)]
        idx = sorted(set.intersection(*[set(zdict[n].index) for n in names]))
        cm = pd.concat([zdict[n].loc[idx, features] for n in names]).groupby(level=0).median()
        for n in names:
            Z = zdict[n].loc[idx].copy()
            Z[features] = Z[features] - cm
            out[n] = Z
    return out


all_z_raw = all_z
all_z = common_mode_removed(all_z_raw, FEATURES)    # run_all() and margins() read the global all_z
r = run_all((0.15, 0.40))
display(margins(r.pivot(index="run", columns="model", values="alarm_life")))
display(r.pivot(index="run", columns="model", values="start_above").round(2))
all_z = all_z_raw                                   # restore

In [ ]:
r = run_all((0.15, 0.40))
r[r["run"].isin(z_runs)].pivot(index="run", columns="model", values="alarm_life").round(3)